## RADAR representation learning

Frozen **wav2vec 2.0**, **HuBERT**, and **Whisper** (encoder only) embeddings with a linear probe (**StandardScaler + LogisticRegression**) and **GroupKFold** (grouped by `participant_id`). Metrics are saved under `results/metrics/repr_learn/`.

RADAR-only notebook — Androids-specific code has been removed.

**Dependencies**: install once in a terminal (`pip install -r requirements.txt`) so you get **transformers 4.x** (`>=4.40,<5`; v5 is untested here). **Do not run `pip install` inside this notebook** — upgrading packages in a live kernel can crash it. After any environment change, use **Restart Kernel** before running model code.

**RADAR audio**: the processed RADAR CSV only stores `File` (e.g. `20201230_1100-scripted-1-1.wav`). `RADAR_AUDIO_ROOT` points at the OneDrive folder containing those `.wav` files (resolved recursively by filename). If OneDrive Files On-Demand is enabled, make sure the folder is set to "Always keep on this device" first, so files are downloaded before this notebook tries to read them.

### 1. Imports

In [16]:
from __future__ import annotations

import gc
import os
from pathlib import Path
from typing import Callable

import re
from datetime import datetime

# Avoid oversubscribing CPU threads (occasionally unstable in Jupyter on Windows)
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1") 

import librosa
import numpy as np
import pandas as pd
import torch

from transformers import (
    HubertModel,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
    WhisperModel,
    WhisperProcessor,
)

### 2. Paths and config

Update these to match your machine and data layout.

In [17]:
PROJECT = Path("/Users/k1777551/JansCode/ASMHI-Research-Project")

# Folder containing all per-site RADAR metadata/PHQ-8 CSVs.
RADAR_CSV_DIR = PROJECT / "data/processed"

# Matches radar_model_dataset_raw_features-KCL.csv, -VUmc.csv, etc.
RADAR_CSV_PATTERN = "radar_model_dataset_raw_features-*.csv"

RESULTS_PATH = PROJECT / "results/metrics/repr_learn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# RADAR .wav files live on OneDrive.
RADAR_AUDIO_ROOT = Path(
    "/Users/k1777551/Library/CloudStorage/OneDrive-King'sCollegeLondon/RADAR-MDD"
)

# --- stability (kernel dies on GPU OOM / Windows DLL mismatch after pip in-notebook) ---
FORCE_CPU = False
BATCH_SIZE = 8

torch.set_num_threads(min(8, max(1, os.cpu_count() or 1)))

if not FORCE_CPU and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

TARGET_SR = 16000
MAX_AUDIO_SECONDS = 30
N_FOLDS = 5

MODEL_IDS = {
    "wav2vec2": "facebook/wav2vec2-base",
    "hubert": "facebook/hubert-base-ls960",
    "whisper": "openai/whisper-base",
}


### 3. Audio helpers

Waveform loading and building a lookup of `.wav` files on disk.

In [18]:
def _mean_pool(hidden: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    # Pools frame-level transformer outputs into one vector per recording.
    if mask is None:
        return hidden.mean(dim=1)
    m = mask.unsqueeze(-1).to(hidden.dtype)
    summed = (hidden * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_wav_filename_index(audio_root: Path) -> dict[tuple[str, str], Path]:
    if not audio_root.exists():
        return {}

    idx: dict[tuple[str, str], Path] = {}

    for p in audio_root.rglob("*.wav"):
        participant_folder = p.parent.name
        idx[(participant_folder, p.name)] = p
        idx[(participant_folder, p.name.lower())] = p

    return idx


def load_waveform_mono(path: Path, target_sr: int) -> np.ndarray:
    wav, sr = librosa.load(str(path), sr=target_sr, mono=True)

    # Currently keeps only the first MAX_AUDIO_SECONDS.
    max_len = int(MAX_AUDIO_SECONDS * target_sr)
    if len(wav) > max_len:
        wav = wav[:max_len]

    return wav.astype(np.float32)

FILENAME_PATTERN = re.compile(
    r"^(?P<date>\d{8})_(?P<time>\d{4})-(?P<rec_type>scripted|unscripted)-\d+(-\d+)?$",
    re.IGNORECASE,
)

def parse_radar_path_metadata(path: Path) -> dict:
    # Expects: .../<site>/<participant_uuid>/<date>_<time>-<scripted|unscripted>-<n>-<n>.wav
    participant_uuid = path.parent.name
    site = path.parent.parent.name

    match = FILENAME_PATTERN.match(path.stem)

    recording_date = None
    recording_time = None
    recording_type = None

    if match:
        try:
            recording_date = datetime.strptime(match.group("date"), "%Y%m%d").date().isoformat()
        except ValueError:
            recording_date = None
        recording_time = match.group("time")
        recording_type = match.group("rec_type").lower()

    return {
        "site": site,
        "participant_id": participant_uuid,
        "recording_date": recording_date,
        "recording_time": recording_time,
        "recording_type": recording_type,
    }


### 4. Encoders

One function per model family, plus alignment back onto the row-level dataframe.

In [19]:
def encode_wav2vec_family(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size

    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)

    if "hubert" in model_id.lower():
        model = HubertModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    else:
        model = Wav2Vec2Model.from_pretrained(model_id, low_cpu_mem_usage=True)

    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []

    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]

        feats = processor(
            waves,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )

        feats = {k: v.to(DEVICE) for k, v in feats.items()}

        with torch.inference_mode():
            outputs = model(**feats)

        pooled = _mean_pool(outputs.last_hidden_state, feats.get("attention_mask"))
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.vstack(out_list)


def encode_whisper(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size

    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []

    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]

        inputs = processor(
            waves,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding="max_length",
        )

        input_features = inputs.input_features.to(DEVICE)

        with torch.inference_mode():
            enc = model.encoder(input_features)
            hidden = enc.last_hidden_state

        pooled = hidden.mean(dim=1)
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.vstack(out_list)


ENCODERS: dict[str, Callable[[list[Path], str, int], np.ndarray]] = {
    "wav2vec2": encode_wav2vec_family,
    "hubert": encode_wav2vec_family,
    "whisper": encode_whisper,
}


def align_embeddings(paths: pd.Series, unique_paths: np.ndarray, mat: np.ndarray) -> np.ndarray:
    # Maps unique audio-level embeddings back onto the row-level dataframe.
    lookup = {str(p): i for i, p in enumerate(unique_paths)}
    idx = np.array([lookup[str(p)] for p in paths], dtype=np.int64)
    return mat[idx]


### 6. Dataset orchestration

Runs every encoder over a dataset's audio, then evaluates with `run_group_cv` and saves CSVs.

In [20]:
def run_repr_for_dataset(
    dataset_key: str,
    manifest: pd.DataFrame,
    path_series: pd.Series,
    groups: pd.Series,
    y: pd.Series,
) -> None:
    valid = (
        path_series.notna()
        & groups.notna()
        & y.notna()
        & path_series.astype(str).str.len().gt(0)
    )

    df = manifest.loc[valid].copy()
    paths = path_series.loc[valid].astype(str)
    yt = y.loc[valid].astype(int).values

    # Parse site / participant ID / date / recording type from the folder structure and filename.
    meta_df = pd.DataFrame([parse_radar_path_metadata(Path(p)) for p in paths])

    uniq = pd.unique(paths)
    uniq_paths = np.array([Path(p) for p in uniq])

    print(f"{dataset_key}: {len(paths)} usable rows | {len(uniq_paths)} unique audio files")

    for model_name, encoder in ENCODERS.items():
        print(f"  -> embeddings: {model_name} ({MODEL_IDS[model_name]})")

        emb_uniq = encoder(uniq_paths.tolist(), MODEL_IDS[model_name])
        X = align_embeddings(paths, uniq, emb_uniq)

        emb_cols = [f"emb_{i}" for i in range(X.shape[1])]
        emb_df = pd.DataFrame(X, columns=emb_cols)

        out_df = pd.DataFrame(
            {
                "audio_path": paths.map(lambda p: Path(p).name).values,
                "label": yt,
            }
        )

        if "phq8_score" in df.columns:
            out_df["phq8_score"] = df["phq8_score"].values

        out_df = pd.concat(
            [
                out_df.reset_index(drop=True),
                meta_df.reset_index(drop=True),
                emb_df.reset_index(drop=True),
            ],
            axis=1,
        )

        out_path = RESULTS_PATH / f"{dataset_key}_{model_name}_embeddings.csv"
        out_df.to_csv(out_path, index=False)

        print(f"     saved: {out_path.name}  ({out_df.shape[0]} rows x {X.shape[1]} embedding dims)")

### 7. RADAR data preparation

Adapt this if your CSV column names or label logic differ.

In [21]:
def prepare_radar(wav_index: dict[str, Path]) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series] | None:

    csv_paths = sorted(RADAR_CSV_DIR.glob(RADAR_CSV_PATTERN))

    if not csv_paths:
        print(f"RADAR: no CSV files found matching '{RADAR_CSV_PATTERN}' in {RADAR_CSV_DIR}")
        return None

    print(f"RADAR: loading {len(csv_paths)} site file(s): {[p.name for p in csv_paths]}")

    df = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)

    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")

    # Only require File and participant_id — rows without a PHQ-8 score are kept
    # (their embeddings still get extracted, just with an empty label downstream).
    df = df.dropna(subset=["File", "participant_id"]).copy()

    # Binary depression threshold. Change/remove if predicting continuous PHQ-8.
    df["depressed"] = (df["phq8_score"] >= 10).astype("Int64")

    names = df["File"].astype(str).str.strip()

    def resolve(participant_id: str, name: str) -> Path | None:
        if not name:
            return None
        key = (participant_id, name)
        if key in wav_index:
            return wav_index[key]
        return wav_index.get((participant_id, name.lower()))

    audio_paths = df.apply(lambda row: resolve(row["participant_id"], str(row["File"]).strip()), axis=1)
    n_ok = audio_paths.notna().sum()

    if n_ok == 0:
        print(
            "RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains",
            "the recording files referenced in column 'File'.",
        )
        return None

    df = df.loc[audio_paths.notna()].copy()
    paths_series = audio_paths.loc[audio_paths.notna()].map(lambda p: Path(p))

    if n_ok < len(names):
        print(f"RADAR: using {n_ok} / {len(names)} rows with found audio files")

    return df, paths_series, df["participant_id"], df["depressed"]


### 8. Run

In [ ]:
RADAR_IDX = build_wav_filename_index(RADAR_AUDIO_ROOT)
rad = prepare_radar(RADAR_IDX)

if rad is not None:
    radar_df, r_paths, r_groups, r_y = rad

    sites = r_paths.map(lambda p: Path(p).parent.parent.name)

    for site_name in sorted(sites.unique()):
        mask = sites == site_name
        site_key = site_name.replace("RADAR-MDD-", "").replace("-s1", "")  # e.g. "KCL"
        run_repr_for_dataset(
            f"radar_{site_key}",
            radar_df.loc[mask],
            r_paths.loc[mask],
            r_groups.loc[mask],
            r_y.loc[mask],
        )

print("Done. Outputs in:", RESULTS_PATH)

RADAR: loading 4 site file(s): ['radar_model_dataset_raw_features-CIBER.csv', 'radar_model_dataset_raw_features-IISPV.csv', 'radar_model_dataset_raw_features-KCL.csv', 'radar_model_dataset_raw_features-VUmc.csv']
RADAR: using 13886 / 13892 rows with found audio files
radar_CIBER: 2086 usable rows | 2086 unique audio files
  -> embeddings: wav2vec2 (facebook/wav2vec2-base)


/opt/anaconda3/envs/jans_project/lib/python3.11/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
